# 💸 Buat Uang di Google Colab

Jalankan **Buat Uang** (generator video otomatis + Clipper) di Google Colab: gratis, lewat browser, tanpa perlu PC khusus.

**Cara pakai:** jalankan sel satu per satu dari atas (klik ▶ di kiri sel, atau `Ctrl+Enter`).

| Langkah | Isi |
|---|---|
| 0 | Aktifkan GPU (opsional, mempercepat Whisper untuk Clipper & subtitle) |
| 1 | Simpan API key di **Secrets** Colab |
| 2 | Instal Buat Uang |
| 3 | Hubungkan Google Drive (opsional, untuk menyimpan hasil & video besar) |
| 4 | Atur konfigurasi |
| 5 | Jalankan WebUI & dapatkan link |
| 6 | Salin hasil ke Google Drive |

> ⚠️ **Jangan bagikan link WebUI ke orang lain.** Siapa pun yang punya link bisa memakai API key-mu. Link berganti setiap kali sel 5 dijalankan ulang.

> ✂️ **Clipper:** gunakan hanya untuk video milikmu sendiri atau video yang pemiliknya sudah memberi izin. Lihat `PANDUAN-CLIPPER.md`.

## 0. Aktifkan GPU (opsional)

Menu **Runtime → Change runtime type → T4 GPU → Save**.

GPU membuat transkripsi Whisper (mode AI Clipper dan subtitle mode whisper) jauh lebih cepat. Untuk membuat video biasa dengan suara Edge TTS, GPU tidak dibutuhkan. Kuota GPU gratis Colab terbatas dan bisa habis.

In [ ]:
!nvidia-smi -L 2>/dev/null || echo "Tidak ada GPU: Whisper akan berjalan di CPU (lebih lambat)."

## 1. Simpan API key di Secrets

Klik ikon **🔑 (Secrets)** di sidebar kiri Colab, lalu tambahkan secret di bawah ini dan aktifkan **Notebook access**. API key disimpan oleh Colab dan tidak tertulis di notebook.

| Nama secret | Wajib? | Didapat dari |
|---|---|---|
| `PEXELS_API_KEY` | Untuk membuat video dari klip stok | https://www.pexels.com/api/ (gratis) |
| `GEMINI_API_KEY` | Jika LLM = `gemini` | https://aistudio.google.com (ada free tier) |
| `DEEPSEEK_API_KEY` | Jika LLM = `deepseek` | https://platform.deepseek.com |
| `OPENAI_API_KEY` | Jika LLM = `openai` | https://platform.openai.com |

Tidak mau pakai API LLM sama sekali? Pilih `none` di langkah 4, lalu tempel naskah dan kata kunci sendiri di WebUI (lihat `PANDUAN-GRATIS-DAN-KUALITAS.md`). Mode AI di Clipper butuh LLM; mode timestamp manual tidak.

## 2. Instal Buat Uang

Butuh beberapa menit. Abaikan peringatan konflik versi dari pip selama sel selesai tanpa `ERROR`.

In [ ]:
#@title Instal { display-mode: "form" }
INSTALL_WHISPER = True  #@param {type:"boolean"}

import os
REPO = "/content/BuatUang"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/cupitebet/BuatUang.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip install -q -r requirements.txt
if INSTALL_WHISPER:
    !pip install -q -r requirements-whisper.txt
print("✅ Instalasi selesai")

## 3. Hubungkan Google Drive (opsional)

File di Colab **hilang saat sesi berakhir**. Dengan Google Drive:
- hasil video bisa disalin ke Drive (langkah 6);
- **video besar untuk Clipper** cukup diunggah sekali ke Drive, lalu di Clipper pilih **File lokal (path)** dan tulis path-nya, misalnya `/content/drive/MyDrive/Podcast/episode-01.mp4`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Atur konfigurasi

In [ ]:
#@title Konfigurasi { display-mode: "form" }
LLM_PROVIDER = "gemini"  #@param ["gemini", "deepseek", "openai", "none"]
GEMINI_MODEL = "gemini-flash-latest"  #@param {type:"string"}
WHISPER_MODEL = "auto"  #@param ["auto", "large-v3", "medium", "small"]

import os, shutil, subprocess
import toml
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

cfg = toml.load(os.path.join(REPO, "config.example.toml"))
app = cfg["app"]

pexels = secret("PEXELS_API_KEY")
app["pexels_api_keys"] = [pexels] if pexels else []

if LLM_PROVIDER != "none":
    key = secret(f"{LLM_PROVIDER.upper()}_API_KEY")
    if not key:
        print(f"⚠️ Secret {LLM_PROVIDER.upper()}_API_KEY belum diisi (langkah 1). Fitur AI tidak akan berjalan.")
    app["llm_provider"] = LLM_PROVIDER
    app[f"{LLM_PROVIDER}_api_key"] = key
    if LLM_PROVIDER == "gemini":
        app["gemini_model_name"] = GEMINI_MODEL

# Sembunyikan panel pengaturan dasar agar API key tidak tampil di WebUI yang dibuka lewat link publik
app["hide_config"] = True

has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
cfg["whisper"]["device"] = "cuda" if has_gpu else "cpu"
cfg["whisper"]["compute_type"] = "float16" if has_gpu else "int8"
cfg["whisper"]["model_size"] = WHISPER_MODEL if WHISPER_MODEL != "auto" else ("large-v3" if has_gpu else "small")

with open(os.path.join(REPO, "config.toml"), "w", encoding="utf-8") as f:
    toml.dump(cfg, f)

print("✅ config.toml dibuat")
print("   Pexels       :", "terisi" if pexels else "KOSONG (wajib untuk video dari klip stok)")
print("   LLM          :", LLM_PROVIDER)
print("   Whisper      :", cfg["whisper"]["model_size"], "di", cfg["whisper"]["device"])

## 5. Jalankan WebUI

Sel ini menjalankan WebUI lalu membuat link publik lewat **Cloudflare Tunnel** (gratis, tanpa akun). Buka link `https://....trycloudflare.com` yang muncul.

- Halaman **Main**: membuat video dari topik/naskah.
- Halaman **Clipper** (sidebar kiri): memotong video panjang jadi klip.

Biarkan tab Colab tetap terbuka. Colab gratis memutus sesi setelah sekitar 90 menit tidak ada interaksi, dan maksimal sekitar 12 jam.

In [ ]:
import re, subprocess, time, urllib.request

CLOUDFLARED = "/content/cloudflared"
if not os.path.exists(CLOUDFLARED):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CLOUDFLARED)
    os.chmod(CLOUDFLARED, 0o755)

for name in ("streamlit_proc", "tunnel_proc"):
    if name in globals() and globals()[name].poll() is None:
        globals()[name].terminate()

webui_log = open("/content/webui.log", "w")
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "webui/Main.py", "--server.port=8501", "--server.headless=true",
     "--browser.gatherUsageStats=false"],
    cwd=REPO, stdout=webui_log, stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read() == b"ok":
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("WebUI gagal start. Cek log: !tail -50 /content/webui.log")

tunnel_log = open("/content/cloudflared.log", "w")
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED, "tunnel", "--no-autoupdate", "--url", "http://localhost:8501"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        url = m.group(0)
        break
    time.sleep(1)

if url:
    print("✅ WebUI siap. Buka link ini (jangan dibagikan):\n")
    print("   " + url)
else:
    print("⚠️ Link belum muncul. Cek log: !tail -50 /content/cloudflared.log")

Kalau ada error saat membuat video, lihat log WebUI:

In [ ]:
!tail -50 /content/webui.log

## 6. Salin hasil ke Google Drive

Jalankan setelah selesai membuat video atau klip (butuh langkah 3). Hasil disalin ke `MyDrive/BuatUang/`.

In [ ]:
import os, shutil
dest = "/content/drive/MyDrive/BuatUang"
if not os.path.isdir("/content/drive/MyDrive"):
    print("Google Drive belum terhubung. Jalankan langkah 3 dulu.")
else:
    copied = 0
    for sub in ("tasks", "clips"):
        src = os.path.join(REPO, "storage", sub)
        if not os.path.isdir(src):
            continue
        for root, _, files in os.walk(src):
            for f in files:
                if f.endswith(".mp4") and not f.startswith(("temp-", "combined-")) and "-raw" not in f:
                    rel = os.path.relpath(os.path.join(root, f), os.path.join(REPO, "storage"))
                    os.makedirs(os.path.dirname(os.path.join(dest, rel)), exist_ok=True)
                    shutil.copy2(os.path.join(root, f), os.path.join(dest, rel))
                    copied += 1
    print(f"✅ {copied} video disalin ke {dest}")